In [1]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib
# matplotlib.use('agg')
import matplotlib.pyplot as plt
# 【关键两行】——必须同时设置！
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'PingFang SC', 'Heiti TC', 'WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False  # 解决负号 '-' 显示为方块的问题[1][2][3][4][5][6][7][8][9][10]

import os
# from tqdm.notebook import  tqdm
from tqdm import  tqdm
import talib
import datetime
import math
import  mplfinance as mpf

import sys
sys.path.append('../../DataSource/聚宽数据')
import datasource 
sys.path.append('../..')
import Utils
# 这个是筛选多少天上涨多少的，
codes = datasource.getAStockCodes()
print(f'股票数量:{len(codes)},第一支股票:{codes[0]}')
code = codes[0]
dt = datasource.getData(code)
dt.head()

股票数量:4929,第一支股票:000001.XSHE


index  open  high   low  close         money  \
code        date                                                       
000001.XSHE 2018-06-20      0  7.55  7.62  7.47   7.59  7.529507e+08   
            2018-06-21      1  7.60  7.68  7.55   7.55  8.360148e+08   
            2018-06-22      2  7.50  7.55  7.48   7.54  6.457529e+08   
            2018-06-25      3  7.59  7.59  7.18   7.24  1.154559e+09   
            2018-06-26      4  7.19  7.23  6.96   7.16  1.042529e+09   

                             volume  
code        date                     
000001.XSHE 2018-06-20   99764990.0  
            2018-06-21  109750599.0  
            2018-06-22   85940350.0  
            2018-06-25  156147915.0  
            2018-06-26  146864886.0

In [3]:
# 这里用kama来
dt.loc[:, 'kama']=talib.KAMA(dt['close'], timeperiod=30) # 这里算自适应均线
dt.tail()

index   open   high    low  close         money  \
code        date                                                          
000001.XSHE 2026-09-07   1995  11.87  11.88  11.65  11.70  1.275606e+09   
            2026-09-08   1996  11.66  11.81  11.65  11.78  8.703611e+08   
            2026-09-09   1997  11.76  11.79  11.69  11.70  6.829468e+08   
            2026-09-10   1998  11.68  11.86  11.66  11.85  1.022544e+09   
            2026-09-11   1999  11.82  11.86  11.71  11.74  9.799906e+08   

                             volume       kama  
code        date                                
000001.XSHE 2026-09-07  108727552.0  11.442941  
            2026-09-08   74051597.0  11.455485  
            2026-09-09   58230598.0  11.461550  
            2026-09-10   86763222.0  11.467276  
            2026-09-11   83246084.0  11.469471

In [6]:
# 然后其他均线
ma_timeperiod = [3,5,10,20,30,60]
for i in ma_timeperiod:
    dt.loc[:, f'MA{i}']=talib.MA(dt['close'], timeperiod=i)
dt.tail()

index   open   high    low  close         money  \
code        date                                                          
000001.XSHE 2026-09-07   1995  11.87  11.88  11.65  11.70  1.275606e+09   
            2026-09-08   1996  11.66  11.81  11.65  11.78  8.703611e+08   
            2026-09-09   1997  11.76  11.79  11.69  11.70  6.829468e+08   
            2026-09-10   1998  11.68  11.86  11.66  11.85  1.022544e+09   
            2026-09-11   1999  11.82  11.86  11.71  11.74  9.799906e+08   

                             volume       kama        MA3     MA5    MA10  \
code        date                                                            
000001.XSHE 2026-09-07  108727552.0  11.442941  11.823333  11.860  11.758   
            2026-09-08   74051597.0  11.455485  11.790000  11.832  11.777   
            2026-09-09   58230598.0  11.461550  11.726667  11.790  11.774   
            2026-09-10   86763222.0  11.467276  11.776667  11.784  11.800   
            2026-09-11   83246084.0  11.469471  11.763333  11.754  11.809   

                           MA20       MA30       MA60  
code        date                                       
000001.XSHE 2026-09-07  11.5120  11.467333  11.051333  
            2026-09-08  11.5380  11.486667  11.063333  
            2026-09-09  11.5605  11.500667  11.076000  
            2026-09-10  11.5905  11.508667  11.093833  
            2026-09-11  11.6220  11.512333  11.114167

In [8]:
# 这里计算一下相关系数，首先提取矩阵
_columns = ['kama']
_columns.extend([f'MA{i}' for i in  ma_timeperiod])
dt2 = dt.loc[:, _columns]
dt2.tail()

kama        MA3     MA5    MA10     MA20  \
code        date                                                        
000001.XSHE 2026-09-07  11.442941  11.823333  11.860  11.758  11.5120   
            2026-09-08  11.455485  11.790000  11.832  11.777  11.5380   
            2026-09-09  11.461550  11.726667  11.790  11.774  11.5605   
            2026-09-10  11.467276  11.776667  11.784  11.800  11.5905   
            2026-09-11  11.469471  11.763333  11.754  11.809  11.6220   

                             MA30       MA60  
code        date                              
000001.XSHE 2026-09-07  11.467333  11.051333  
            2026-09-08  11.486667  11.063333  
            2026-09-09  11.500667  11.076000  
            2026-09-10  11.508667  11.093833  
            2026-09-11  11.512333  11.114167

In [9]:
dt2.corr() # 相关系数

,kama,MA3,MA5,MA10,MA20,MA30,MA60
kama,1.000000,0.974522,0.978520,0.986374,0.995194,0.997407,0.984062
MA3,0.974522,1.000000,0.998893,0.993548,0.982380,0.971686,0.939856
MA5,0.978520,0.998893,1.000000,0.996702,0.986269,0.975810,0.944181
MA10,0.986374,0.993548,0.996702,1.000000,0.993885,0.984620,0.954162
MA20,0.995194,0.982380,0.986269,0.993885,1.000000,0.996148,0.970864
MA30,0.997407,0.971686,0.975810,0.984620,0.996148,1.000000,0.983594
MA60,0.984062,0.939856,0.944181,0.954162,0.970864,0.983594,1.000000


看起来，kama跟20日均线和30日均线的相关系数最高啊。

In [11]:
# 这里做一下全局的
_corrs = {} # 所有股票的相关系数
ma_timeperiod = [3,5,10,20,30,60]
for i in tqdm(range(len(codes))):
    # 所有股票的，我这里知识取得相关系统
    code = codes[i]
    dt = datasource.getData(code)
    dt.loc[:, 'kama']=talib.KAMA(dt['close'], timeperiod=30) # 这里算自适应均线
    # 计算均线
    for j in ma_timeperiod:
        dt.loc[:, f'MA{j}']=talib.MA(dt['close'], timeperiod=j)
    # 筛选相关列
    _columns = ['kama']
    _columns.extend([f'MA{i}' for i in  ma_timeperiod])
    dt2 = dt.loc[:, _columns] # 只是取得特定列
    # 保存相关系统
    _corrs[code]=dt2.corr()

100%|██████████████████████████████████████████████████████████████████████████████| 4929/4929 [01:35<00:00, 51.36it/s]


In [16]:
_corrs['000001.XSHE']

,kama,MA3,MA5,MA10,MA20,MA30,MA60
kama,1.000000,0.974522,0.978520,0.986374,0.995194,0.997407,0.984062
MA3,0.974522,1.000000,0.998893,0.993548,0.982380,0.971686,0.939856
MA5,0.978520,0.998893,1.000000,0.996702,0.986269,0.975810,0.944181
MA10,0.986374,0.993548,0.996702,1.000000,0.993885,0.984620,0.954162
MA20,0.995194,0.982380,0.986269,0.993885,1.000000,0.996148,0.970864
MA30,0.997407,0.971686,0.975810,0.984620,0.996148,1.000000,0.983594
MA60,0.984062,0.939856,0.944181,0.954162,0.970864,0.983594,1.000000


In [17]:
type(_corrs['000001.XSHE'])

pandas.DataFrame

In [26]:
# 接下来做平均,我这里用麻烦的方式来处理，首先取得列名和行名
_columns2 = _corrs['000001.XSHE'].columns.to_list()
print(_columns2)
_row2 = _corrs['000001.XSHE'].index.to_list()
print(_row2)


['kama', 'MA3', 'MA5', 'MA10', 'MA20', 'MA30', 'MA60']
['kama', 'MA3', 'MA5', 'MA10', 'MA20', 'MA30', 'MA60']


In [28]:
# 构造DataFrmae需要的字典
df_list = list(_corrs.values())
df_list = [df.sort_index().sort_index(axis=1) for df in df_list]  # 可选：标准化索引/列
arr_3d = np.array([df.to_numpy() for df in df_list])  # shape: (k, n_rows, n_cols)
mean_3d = arr_3d.mean(axis=0)  # shape: (n_rows, n_cols)，即每位置的跨实验均值
mean_3d

array([[1.        , 0.98753222, 0.98675465, 0.96852934, 0.99319062,
        0.91374088, 0.97660621],
       [0.98753222, 1.        , 0.96424968, 0.99192841, 0.97218466,
        0.94404604, 0.98884983],
       [0.98675465, 0.96424968, 1.        , 0.94303865, 0.99756012,
        0.88836342, 0.95697247],
       [0.96852934, 0.99192841, 0.94303865, 1.        , 0.95115156,
        0.96794884, 0.98874221],
       [0.99319062, 0.97218466, 0.99756012, 0.95115156, 1.        ,
        0.89603066, 0.96392075],
       [0.91374088, 0.94404604, 0.88836342, 0.96794884, 0.89603066,
        1.        , 0.95521414],
       [0.97660621, 0.98884983, 0.95697247, 0.98874221, 0.96392075,
        0.95521414, 1.        ]])

In [30]:
df_corrs = pd.DataFrame(mean_3d,columns=_columns2, index=_row2)
df_corrs

,kama,MA3,MA5,MA10,MA20,MA30,MA60
kama,1.000000,0.987532,0.986755,0.968529,0.993191,0.913741,0.976606
MA3,0.987532,1.000000,0.964250,0.991928,0.972185,0.944046,0.988850
MA5,0.986755,0.964250,1.000000,0.943039,0.997560,0.888363,0.956972
MA10,0.968529,0.991928,0.943039,1.000000,0.951152,0.967949,0.988742
MA20,0.993191,0.972185,0.997560,0.951152,1.000000,0.896031,0.963921
MA30,0.913741,0.944046,0.888363,0.967949,0.896031,1.000000,0.955214
MA60,0.976606,0.988850,0.956972,0.988742,0.963921,0.955214,1.000000


总体上kama跟20日均线日均线最接近。